In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score,mean_absolute_error, mean_squared_error
from transformers import RobertaTokenizer, RobertaModel
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
from collections import Counter
from imblearn.over_sampling import RandomOverSampler
from torch.utils.data import DataLoader, Subset
from scipy.stats import pearsonr
from tqdm import tqdm
from sklearn.exceptions import FitFailedWarning
import warnings
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV
import matplotlib.pyplot as plt
from scipy.stats import norm
import matplotlib
from sklearn.base import clone


In [2]:
import torch

# 检查是否有可用的 GPU
if torch.cuda.is_available():
    print("CUDA 可用，GPU 可用。")
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 数量: {torch.cuda.device_count()}")
    print(f"当前设备名称: {torch.cuda.get_device_name(0)}")
    print(f"当前设备总内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("CUDA 不可用，仅支持 CPU。")


CUDA 可用，GPU 可用。
CUDA 版本: 12.1
GPU 数量: 1
当前设备名称: NVIDIA GeForce RTX 3080 Ti
当前设备总内存: 11.63 GB


In [3]:
import pandas as pd
from rdkit import Chem

In [4]:
data=pd.read_excel("./Worms_unique.xlsx")
data=data.dropna()
smiles_data = data['SMILES_Canonical_RDKit'].tolist()
mgperL = data['mgperL'].values
Duration_Value= data['Duration_Value'].values

In [5]:
endpoint = data['endpoint']
mgperL=np.log1p(mgperL)

In [6]:
# 数据增强函数：简单的 SMILES 序列翻转
def augment_smiles(smiles):
    """简单的数据增强方法，例如旋转 SMILES 字符串"""
    if random.random() > 0.5:
        return smiles[::-1]  # 翻转字符串
    return smiles

# 根据 mgperL 浓度生成分类标签
def generate_labels(mgperL):
    """根据 mgperL 的浓度范围生成分类标签"""
    if mgperL < 0.2:
        return 0  # high
    elif 0.2 <= mgperL < 1.5:
        return 1  # mid
    elif 1.5 <= mgperL < 3.5:
        return 2  # mid
    else:
        return 3  # low

# 数据集定义
class SMILES_Dataset(Dataset):
    def __init__(self, smiles, reg_labels, class_labels):
        self.smiles = smiles
        self.reg_labels = reg_labels  # 回归任务标签 (mgperL)
        self.class_labels = class_labels  # 分类任务标签

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        #smiles = augment_smiles(self.smiles[idx])  # 数据增强
        smiles = self.smiles[idx]  # 数据增强
        reg_label = self.reg_labels[idx]
        class_label = self.class_labels[idx]
        tokens = tokenizer(smiles, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
        return tokens, torch.tensor(reg_label, dtype=torch.float32), torch.tensor(class_label, dtype=torch.long)



class ChemBERTa_MultiTask(nn.Module):
    def __init__(self, num_classes):
        super(ChemBERTa_MultiTask, self).__init__()
        # 加载预训练的ChemBERTa模型
        self.chemberta = chemberta_model
        hidden_size = self.chemberta.config.hidden_size  # 一般为768
        # 回归任务的全连接层
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

        # 分类任务的全连接层
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, 4)
        )



    def forward(self, tokens):
        output = self.chemberta(**tokens)
        cls_embedding = output.last_hidden_state[:, 0, :]  # [CLS] token 嵌入
        reg_output = self.regressor(cls_embedding)  # 回归任务输出
        class_output = self.classifier(cls_embedding)  # 分类任务输出
        return reg_output, class_output

In [7]:
# 加载 ChemBERTa 模型和 tokenizer
model_name = "../models/chemBERTa"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
chemberta_model = RobertaModel.from_pretrained(model_name)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ChemBERTa_MultiTask(chemberta_model).to(device)  
# 加载最佳模型
model.load_state_dict(torch.load('./Worms_model.pth'))

/root/miniconda3/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


<All keys matched successfully>

In [8]:
# 提取 SMILES 的嵌入表示
def extract_embeddings(smiles_list, model, tokenizer, device):
    embeddings = []
    model.eval()  # 设置模型为评估模式
    with torch.no_grad():
        for smiles in smiles_list:
            # Tokenize the SMILES string
            tokens = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
            
            # 通过 ChemBERTa 模型获得输出
            outputs = model.chemberta(**tokens)  # 提取 ChemBERTa 模型的输出
            
            # 提取 [CLS] token 的嵌入作为 SMILES 的整体嵌入
            cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # 获取多维的 [CLS] 嵌入
            embeddings.append(cls_embedding)
    
    return np.vstack(embeddings)  # 将所有嵌入拼接成一个矩阵

In [9]:
#plot_pca_with_variance(pca_result, labels, explained_variance)

In [10]:
# 提取嵌入
smiles_embeddings = extract_embeddings(smiles_data, model, tokenizer, device)

In [11]:
import xgboost as xgb

In [12]:
# 检查需要One-Hot编码的列，并进行编码（如果类别超过一种）
def encode_column(data, column_name):
    unique_values = data[column_name].unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(data[[column_name]])
    else:
        return None  # 只有一种类别时忽略

# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(data, 'effect')
endpoint_encoded = encode_column(data, 'endpoint')
species_encoded = encode_column(data, 'species_group')

# 将需要的列拼接成输入 X
X = np.hstack((smiles_embeddings, data['Duration_Value'].values.reshape(-1, 1)))

# 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded, species_encoded]:
    if encoded_feature is not None:
        X = np.hstack((X, encoded_feature))

# 目标值 y
y = mgperL

In [13]:
import os

In [14]:
import joblib

In [15]:
 {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.8}

{'subsample': 0.8,
 'n_estimators': 300,
 'max_depth': 5,
 'learning_rate': 0.01,
 'colsample_bytree': 0.8}

In [16]:

# 2. 设置最佳超参数
best_params_xgb = {
    'n_estimators': 300,
    'subsample': 0.8,
    'max_depth': 5,
    'learning_rate': 0.01,
    'colsample_bytree': 0.8
}

# 3. 创建 XGBoost 模型
# 初始化并训练模型
xgb_model = XGBRegressor(n_jobs=-1, verbosity=0, **best_params_xgb)
xgb_model.fit(X, y)



model_path = os.path.join('Worms_xgb_single.json')
xgb_model.save_model(model_path)
print(f"💾 XGBoost 模型已保存至: {model_path}")

💾 XGBoost 模型已保存至: Worms_xgb_single.json


In [17]:
#预测危险化合物

In [18]:
IECSC = pd.read_excel("../IECSC.xlsx")

In [19]:
IECSC_SMILES =  IECSC['SMILES'].tolist()

In [20]:
IECSC_smiles_embeddings = extract_embeddings(IECSC_SMILES, model, tokenizer, device)

In [24]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
import torch  # 只用来读 state_dict，推断训练时 EXTRA_DIM

# ----------------- 路径与固定取值 -----------------
TRAIN_EXCEL_PATH = "./Worms_unique.xlsx"  # 用于拟合 OneHot

                       # 仅用于保存时带上原 SMILES

BERT_WEIGHTS     = "./Worms_model.pth"                  # 只为推断训练时 EXTRA_DIM
XGB_MODEL_PATH   = "./Worms_xgb_single.json"   # 训练好的 XGB 模型
SAVE_DIR         = "./predictions"; os.makedirs(SAVE_DIR, exist_ok=True)

# 固定的端点/效应/物种/时长（按需改）
FIXED_ENDPOINT = "LC50"
FIXED_EFFECT   = "MOR"
FIXED_SPECIES  = 'Worms'
FIXED_DURATION = 336.0  # 小时

# ----------------- 读取嵌入（若已在内存，直接赋值） -----------------
emb_cls = IECSC_smiles_embeddings.astype(np.float32)
N, H = emb_cls.shape
assert H > 0, "Empty embeddings."

# ----------------- 基于训练表拟合 One-Hot（与训练编码空间一致） -----------------

# ================== 基于训练表拟合 One-Hot（与训练编码空间一致） ==================
train_df = pd.read_excel(TRAIN_EXCEL_PATH).dropna(subset=[
    "SMILES_Canonical_RDKit", "Duration_Value", "effect", "endpoint", "species_group"
])

def fit_optional_ohe(df, col):
    vals = df[col].dropna().unique()
    if len(vals) > 1:
        enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(df[[col]])
        print(f"[OHE] {col}: {len(enc.categories_[0])} levels -> {list(enc.categories_[0])[:8]}{' ...' if len(enc.categories_[0])>8 else ''}")
        return enc
    print(f"[OHE] {col}: 单一类别（训练时未做 One-Hot）")
    return None

enc_effect   = fit_optional_ohe(train_df, "effect")
enc_endpoint = fit_optional_ohe(train_df, "endpoint")


extra_dim_by_train_logic = 1 \
    + (enc_effect.categories_[0].size   if enc_effect   else 0) \
    + (enc_endpoint.categories_[0].size if enc_endpoint else 0) 

print(f"Extra feature dim (by training logic) = {extra_dim_by_train_logic}")

# ================== 构造额外特征：duration +（effect/endpoint/species 的 One-Hot） ==================
parts = [np.full((N, 1), float(FIXED_DURATION), dtype=np.float32)]

if enc_effect is not None:
    eff = enc_effect.transform(pd.DataFrame({"effect": [FIXED_EFFECT]*N})).astype(np.float32)
    parts.append(eff)

if enc_endpoint is not None:
    endp = enc_endpoint.transform(pd.DataFrame({"endpoint": [FIXED_ENDPOINT]*N})).astype(np.float32)
    parts.append(endp)



extra_feats = np.hstack(parts) if parts else np.zeros((N, 0), dtype=np.float32)
print(f"Extra features: {extra_feats.shape}")


[OHE] effect: 5 levels -> ['BEH', 'ENZ', 'ITX', 'MOR', 'POP']
[OHE] endpoint: 5 levels -> ['EC10', 'EC50', 'LC50', 'LOEC', 'LT50']
Extra feature dim (by training logic) = 11
Extra features: (26646, 11)


In [25]:

# ----------------- 拼接并用 XGBoost 预测（训练标签为 log1p(mg/L)） -----------------
X_new = np.hstack([emb_cls, extra_feats]).astype(np.float32)  # [N, H + EXTRA_DIM_CKPT]

# xgb_model = XGBRegressor()
# xgb_model.load_model(XGB_MODEL_PATH)

y_log = xgb_model.predict(X_new).astype(np.float32)
y_mgL = np.expm1(y_log)

# ----------------- 保存结果（带上原始 SMILES 以便追溯） -----------------
def pick_smiles(df):
    for n in ["SMILES_Canonical_RDKit", "SMILES", "smiles"]:
        if n in df.columns: return n
    return None

try:
    raw = pd.read_excel(NEW_EXCEL_PATH)
    c_smiles = pick_smiles(raw)
    smiles = raw[c_smiles].astype(str).tolist() if c_smiles else [None]*N
    if len(smiles) != N:
        print(f"⚠️ 提示：SMILES 行数({len(smiles)})与嵌入数({N})不一致，结果中将按索引保存。")
        smiles = [None]*N
except Exception:
    smiles = [None]*N  # 如果新表不在或列名不匹配，也不阻塞

out = pd.DataFrame({
    "SMILES": IECSC_SMILES,
    "endpoint_fixed": FIXED_ENDPOINT,
    "effect_fixed":   FIXED_EFFECT,
    "duration_fixed": FIXED_DURATION,
    "species_fixed":  FIXED_SPECIES,
    "pred_log_mgperL": y_log,
    "pred_mgperL":     y_mgL,
})

fname = f"Worms_{FIXED_ENDPOINT}_{FIXED_EFFECT}_{int(FIXED_DURATION)}h.xlsx"
save_path = os.path.join(SAVE_DIR, fname)
out.to_excel(save_path, index=False)
print(f"✅ 预测完成，已保存：{save_path}")

✅ 预测完成，已保存：./predictions/Worms_LC50_MOR_336h.xlsx


In [26]:
# 固定的端点/效应/物种/时长（按需改）
FIXED_ENDPOINT = "EC10"
FIXED_EFFECT   = "REP"
FIXED_SPECIES  = 'Worms'
FIXED_DURATION = 1344.0  # 小时

# ----------------- 读取嵌入（若已在内存，直接赋值） -----------------
emb_cls = IECSC_smiles_embeddings.astype(np.float32)
N, H = emb_cls.shape
assert H > 0, "Empty embeddings."

# ----------------- 基于训练表拟合 One-Hot（与训练编码空间一致） -----------------

# ================== 基于训练表拟合 One-Hot（与训练编码空间一致） ==================
train_df = pd.read_excel(TRAIN_EXCEL_PATH).dropna(subset=[
    "SMILES_Canonical_RDKit", "Duration_Value", "effect", "endpoint", "species_group"
])

def fit_optional_ohe(df, col):
    vals = df[col].dropna().unique()
    if len(vals) > 1:
        enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(df[[col]])
        print(f"[OHE] {col}: {len(enc.categories_[0])} levels -> {list(enc.categories_[0])[:8]}{' ...' if len(enc.categories_[0])>8 else ''}")
        return enc
    print(f"[OHE] {col}: 单一类别（训练时未做 One-Hot）")
    return None

enc_effect   = fit_optional_ohe(train_df, "effect")
enc_endpoint = fit_optional_ohe(train_df, "endpoint")


extra_dim_by_train_logic = 1 \
    + (enc_effect.categories_[0].size   if enc_effect   else 0) \
    + (enc_endpoint.categories_[0].size if enc_endpoint else 0) 

print(f"Extra feature dim (by training logic) = {extra_dim_by_train_logic}")

# ================== 构造额外特征：duration +（effect/endpoint/species 的 One-Hot） ==================
parts = [np.full((N, 1), float(FIXED_DURATION), dtype=np.float32)]

if enc_effect is not None:
    eff = enc_effect.transform(pd.DataFrame({"effect": [FIXED_EFFECT]*N})).astype(np.float32)
    parts.append(eff)

if enc_endpoint is not None:
    endp = enc_endpoint.transform(pd.DataFrame({"endpoint": [FIXED_ENDPOINT]*N})).astype(np.float32)
    parts.append(endp)



extra_feats = np.hstack(parts) if parts else np.zeros((N, 0), dtype=np.float32)
print(f"Extra features: {extra_feats.shape}")
# ----------------- 拼接并用 XGBoost 预测（训练标签为 log1p(mg/L)） -----------------
X_new = np.hstack([emb_cls, extra_feats]).astype(np.float32)  # [N, H + EXTRA_DIM_CKPT]

# xgb_model = XGBRegressor()
# xgb_model.load_model(XGB_MODEL_PATH)

y_log = xgb_model.predict(X_new).astype(np.float32)
y_mgL = np.expm1(y_log)

# ----------------- 保存结果（带上原始 SMILES 以便追溯） -----------------
def pick_smiles(df):
    for n in ["SMILES_Canonical_RDKit", "SMILES", "smiles"]:
        if n in df.columns: return n
    return None

try:
    raw = pd.read_excel(NEW_EXCEL_PATH)
    c_smiles = pick_smiles(raw)
    smiles = raw[c_smiles].astype(str).tolist() if c_smiles else [None]*N
    if len(smiles) != N:
        print(f"⚠️ 提示：SMILES 行数({len(smiles)})与嵌入数({N})不一致，结果中将按索引保存。")
        smiles = [None]*N
except Exception:
    smiles = [None]*N  # 如果新表不在或列名不匹配，也不阻塞

out = pd.DataFrame({
    "SMILES": IECSC_SMILES,
    "endpoint_fixed": FIXED_ENDPOINT,
    "effect_fixed":   FIXED_EFFECT,
    "duration_fixed": FIXED_DURATION,
    "species_fixed":  FIXED_SPECIES,
    "pred_log_mgperL": y_log,
    "pred_mgperL":     y_mgL,
})

fname = f"Worms_{FIXED_ENDPOINT}_{FIXED_EFFECT}_{int(FIXED_DURATION)}h.xlsx"
save_path = os.path.join(SAVE_DIR, fname)
out.to_excel(save_path, index=False)
print(f"✅ 预测完成，已保存：{save_path}")

[OHE] effect: 5 levels -> ['BEH', 'ENZ', 'ITX', 'MOR', 'POP']
[OHE] endpoint: 5 levels -> ['EC10', 'EC50', 'LC50', 'LOEC', 'LT50']
Extra feature dim (by training logic) = 11
Extra features: (26646, 11)
✅ 预测完成，已保存：./predictions/Worms_EC10_REP_1344h.xlsx
